# Gap & Stone-Separation Metrics (Step 5)

Quantifies what pixel IoU cannot: whether models **separate individual stones**
or merge them, and how well the **inter-stone gaps** are detected. Port of the
archived `04_segmentation_evaluation` notebooks; see `amg_pipeline/gapmetrics.py`.

Per (experiment, variant, run, wall) + pooled AllWalls row:
- **gap metrics**: gap IoU/precision/recall/F1 + predicted/GT gap-pixel ratio
  (>1.2 over-segments, <0.8 under-segments), gaps via closing (disk r=45);
- **stone metrics**: per-GT-stone coverage & separation; detected/merged/
  undetected counts at threshold pairs 0.9/0.9 (primary, as in the archived
  notebook), 0.7/0.7, 0.5/0.5; per-class breakdown at 0.9/0.9.

**Run on the Windows machine** (reads GT masks + RAW rasters; eval-only, CPU).
Runtime is dominated by one morphological closing per raster: the default set
(2 ensembles + 5-run frozen baseline = 84 rasters) ≈ 30–90 min. Progress is
checkpointed per experiment to a .partial.csv; a completed run writes
`experiments/v9_gapstone/gap_stone_metrics.csv` (pull this back).

## 0. Imports

In [ ]:
import os, sys, dataclasses
import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import amg_pipeline as amg
from amg_pipeline.config import RunConfig
from amg_pipeline.gapmetrics import run_gap_eval
from amg_pipeline import paths

print("amg_pipeline loaded from:", os.path.dirname(amg.__file__))

## 1. CONFIG

In [ ]:
EXPERIMENTS_ROOT = os.path.join(REPO_ROOT, "experiments")
OUT_NAME = "v9_gapstone"

# Which rasters to evaluate: (experiment_name, n_runs)
INCLUDE_COSINE_ENS = True      # v8_coslr_ens for completeness (12 rasters)
INCLUDE_FROZEN_BASELINE = True # per-run spread context (60 rasters, the slow part)
EXPERIMENTS = [("v2_baseline_ens", 1)]
if INCLUDE_COSINE_ENS:
    EXPERIMENTS.append(("v8_coslr_ens", 1))
if INCLUDE_FROZEN_BASELINE:
    EXPERIMENTS.append(("v2_yaw_correction-epsV1", 5))

CHANNEL_VARIANTS = (3, 4, 7)
KERNEL_RADIUS = 45            # same closing radius as the ROI evaluation
MIN_STONE_SIZE = 100          # px, as in the archived notebook

TEST_MASK_DIR = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\03_test-masks"
WALLS = ["wall1", "wall2", "wall3", "wall4"]

BASE_CONFIG = RunConfig(
    channels=7, run_number=1,
    experiment_name=EXPERIMENTS[0][0], experiments_root=EXPERIMENTS_ROOT,
    test_mask_dir=TEST_MASK_DIR, walls=WALLS,
    kernel_radius=KERNEL_RADIUS,
)
CSV_PATH = os.path.join(EXPERIMENTS_ROOT, OUT_NAME, "gap_stone_metrics.csv")
print("experiments:", EXPERIMENTS)
print("will write ->", CSV_PATH)

## 2. Run (eval-only; skip-if-exists on the final CSV)

In [ ]:
DO_RUN = True    # <-- armed
FORCE = False

if DO_RUN:
    gs = run_gap_eval(BASE_CONFIG, EXPERIMENTS, channel_variants=CHANNEL_VARIANTS,
                      out_name=OUT_NAME, kernel_radius=KERNEL_RADIUS,
                      min_stone_size=MIN_STONE_SIZE, force=FORCE)
else:
    print("DO_RUN is False")

## 3. Summary — the pixel-vs-stone question

AllWalls (pooled stones / summed gap counts) per experiment and variant.
The hypothesis under test: appearance-only wins pixel metrics but merges
stones; geometry-only separates cleanly; the full model sits between.

In [ ]:
if not os.path.exists(CSV_PATH):
    print("No CSV yet."); gs = None
else:
    gs = pd.read_csv(CSV_PATH)
    aw = gs[gs.wall == "AllWalls"]
    cols = ["gap_iou", "gap_f1", "gap_ratio", "detection_rate_c90_s90",
            "merge_rate_c90_s90", "detection_rate_c50_s50", "mean_separation"]
    pd.set_option("display.width", 220)

    for exp in aw.experiment.unique():
        sub = aw[aw.experiment == exp]
        print(f"\n=== {exp} ===")
        if sub.run_number.nunique() > 1:
            t = sub.groupby("channels")[cols].agg(["mean", "std"]).round(4)
        else:
            t = sub.set_index("channels")[cols].round(4)
        display(t)

## 4. Per-class stone separation (final ensemble) — the ashlar question

In [ ]:
if gs is not None:
    ens = gs[(gs.experiment == "v2_baseline_ens") & (gs.wall == "AllWalls")]
    rows = []
    for _, r in ens.iterrows():
        for cls in ("Ashlar", "Polygonal", "Quarry"):
            if f"{cls}_n" in r and not pd.isna(r.get(f"{cls}_n")):
                rows.append({"channels": r.channels, "class": cls,
                             "n": int(r[f"{cls}_n"]),
                             "detection_rate": r[f"{cls}_detection_rate_c90_s90"],
                             "merge_rate": r[f"{cls}_merge_rate_c90_s90"],
                             "mean_coverage": r[f"{cls}_mean_coverage"],
                             "mean_separation": r[f"{cls}_mean_separation"]})
    t = pd.DataFrame(rows).pivot(index="class", columns="channels",
                                 values=["detection_rate", "merge_rate", "mean_separation"]).round(3)
    display(t)
    print("\nReading: high merge_rate + low mean_separation = stones fused together.")
    print("If 4ch ashlar shows this while 3ch does not, the visual impression is confirmed.")